# Variant A — Baseline CNN training

Transfer learning (EfficientNet-B0, ImageNet-pretrained) + class-weighted cross-entropy loss, on IDRiD Disease Grading.
This is the control variant everything else in the ablation study gets compared against.

**Before running:** make sure the repo has been pushed to GitHub (it's public, no login needed to clone). Runtime > Change runtime type > GPU.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Dilshan-Fernando-01/Computer-Vision-Assignment.git
%cd Computer-Vision-Assignment

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python3 scripts/download_dataset.py

In [ ]:
!python3 src/datasets/splits.py

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from torch.utils.data import DataLoader

from src.datasets.dataset import IDRiDGradingDataset, IMAGES_DIR_TRAIN, IMAGES_DIR_TEST
from src.augmentation.augment import get_training_augmentations, compute_class_weights
from src.models.build_model import build_model
from src.training.train import fit, evaluate, get_device

DEVICE = get_device()
IMAGE_SIZE = 512
BATCH_SIZE = 16
EPOCHS = 40
PATIENCE = 8
LR = 1e-4

print('device:', DEVICE)

In [ ]:
train_ds = IDRiDGradingDataset('data/processed/splits/train.csv', IMAGES_DIR_TRAIN, image_size=IMAGE_SIZE, transform=get_training_augmentations(IMAGE_SIZE))
val_ds   = IDRiDGradingDataset('data/processed/splits/val.csv',   IMAGES_DIR_TRAIN, image_size=IMAGE_SIZE)
test_ds  = IDRiDGradingDataset('data/processed/splits/test.csv',  IMAGES_DIR_TEST,  image_size=IMAGE_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=2)

train_labels = [label for _, label in train_ds.samples]
class_weights = compute_class_weights(train_labels)
print('class weights:', class_weights)

criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))
model = build_model('efficientnet_b0', num_classes=5, pretrained=True)

In [ ]:
history = fit(
    model, train_loader, val_loader, criterion,
    epochs=EPOCHS, lr=LR, patience=PATIENCE,
    checkpoint_path='outputs/checkpoints/variant_a_efficientnet_b0.pt',
    device=DEVICE,
)

In [ ]:
import json
import os

os.makedirs('outputs/history', exist_ok=True)
with open('outputs/history/variant_a_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('best val macro F1:', history['best_val_macro_f1'])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()

axes[2].plot(history['val_macro_f1'], label='val macro F1', color='green')
axes[2].set_title('Validation Macro F1'); axes[2].set_xlabel('epoch'); axes[2].legend()

fig.suptitle('Variant A - Baseline EfficientNet-B0')
fig.tight_layout()
fig.savefig('outputs/history/variant_a_curves.png', dpi=120)
plt.show()

## Final evaluation on the held-out test set (loads the best checkpoint, by val macro F1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.load_state_dict(torch.load('outputs/checkpoints/variant_a_efficientnet_b0.pt', map_location=DEVICE))
test_metrics = evaluate(model, test_loader, criterion, DEVICE)

print('Test accuracy:', test_metrics['accuracy'])
print('Test macro F1:', test_metrics['macro_f1'])
print()
print(classification_report(test_metrics['labels'], test_metrics['preds'], target_names=[f'Stage {i}' for i in range(5)]))
print('Confusion matrix (rows=true, cols=predicted):')
print(np.array(confusion_matrix(test_metrics['labels'], test_metrics['preds'])))